# Week 3 — Game Theory Implementation
**Project:** Event-Driven 6G Scheduling using SimPy + Sionna  
**Group 4 | Summer Research Internship 2025**

This notebook explores the payoff functions and helps understand how Ud and Ua change with different attack/defend scenarios.

In [ ]:
import sys
import os
sys.path.insert(0, '../week3')

import numpy as np
import matplotlib.pyplot as plt
from payoff_engine import Ud, Ua, P_detect, IP

print('Payoff engine imported successfully')

## 1. Defender vs Attacker Utility

In [ ]:
# Vary D and C from 0 to 5
print(f"{'D':>4} {'C':>4} {'Ud':>8} {'Ua':>8} {'Winner':>10}")
print('-' * 40)
for D, C in [(5,0),(4,1),(3,2),(2,3),(1,4),(0,5)]:
    ud = Ud(D, C)
    ua = Ua(C, D)
    winner = 'Defender' if ud > ua else 'Attacker'
    print(f"{D:>4} {C:>4} {ud:>8.2f} {ua:>8.2f} {winner:>10}")

## 2. Detection Probability Curve

In [ ]:
risk_vals = np.linspace(0, 1, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Effect of risk score
for F in [0.0, 0.3, 0.6, 0.9]:
    pd = [P_detect(r, F) for r in risk_vals]
    axes[0].plot(risk_vals, pd, label=f'Fatigue={F}')
axes[0].set_title('Detection Probability vs Risk Score', fontweight='bold')
axes[0].set_xlabel('Risk Score (Ri)')
axes[0].set_ylabel('P_detect')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Effect of fatigue
fatigue_vals = np.linspace(0, 1, 100)
for Ri in [0.3, 0.5, 0.7, 0.9]:
    pd = [P_detect(Ri, f) for f in fatigue_vals]
    axes[1].plot(fatigue_vals, pd, label=f'Risk={Ri}')
axes[1].set_title('Detection Probability vs Defender Fatigue', fontweight='bold')
axes[1].set_xlabel('Fatigue (F)')
axes[1].set_ylabel('P_detect')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Week 3 — P_detect Analysis', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/pdetect_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pdetect_analysis.png')

## 3. Utility Surface — Ud over all D, C combinations

In [ ]:
D_vals = np.arange(0, 11)
C_vals = np.arange(0, 11)
Ud_grid = np.array([[Ud(d, c) for c in C_vals] for d in D_vals])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(Ud_grid, cmap='RdYlGn', aspect='auto', origin='lower')
ax.set_xlabel('Compromised (C)', fontsize=11)
ax.set_ylabel('Detected (D)', fontsize=11)
ax.set_title('Defender Utility Ud = α·D − β·C\n(Green = Defender Winning, Red = Attacker Winning)', fontweight='bold')
ax.set_xticks(range(11)); ax.set_yticks(range(11))
fig.colorbar(im, label='Ud Value')
plt.tight_layout()
plt.savefig('../results/ud_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ud_surface.png')

## 4. Key Insight Summary

In [ ]:
print('Key Insights from Payoff Analysis:')
print()
print('1. Ud is positive only when D > 2C (since beta=2)')
print('   -> Defender must detect more than twice the compromises to have positive utility')
print()
print('2. P_detect increases with risk score but decreases with fatigue')
print('   -> High-risk nodes are easier to detect — defender should monitor them')
print()
print('3. Stackelberg: defender picks monitoring set to maximise Ud')
print('   knowing attacker will pick the target that maximises Ua')
print()
print('4. Nash: both players find equilibrium simultaneously')
print('   -> No anticipation, just stable mutual best response')